# Synthetic Stress Test Data Generator (Hybrid GRU + Jump-Diffusion)

### Architectural Disclaimer: The Flaws of Raw GRUs in Financial Generation
This notebook demonstrates a hybrid approach to generating synthetic financial time series. It is important to document that using a **raw GRU (or LSTM)** as a purely generative engine for financial data is fundamentally flawed. 

* **Momentum vs. Regime:** Neural networks like GRUs are exceptionally good at identifying short-term conditional momentum (e.g., "If the last 4 days looked like X, day 5 usually looks like Y"). However, they are completely blind to macro-economic **regimes** (bull markets, bear markets, high-volatility environments). 
* **Autoregressive Collapse:** If left to its own devices, a generative GRU feeding its own predictions back into itself will rapidly suffer from autoregressive collapse. It will either flatline to the historical mean (a straight line) or enter an explosive momentum feedback loop (spiraling to zero or infinity). 

**The Solution Used Here:** To make the GRU viable, we do not let it generate data in a vacuum. Instead, we use a **Mixture Model**. The GRU handles the standard day-to-day momentum, while an external Stochastic Engine (Brownian motion + Lognormal news shocks) forces regime changes and volatility. We also fit a **Dual-Tail Generalized Pareto Distribution (GPD)** to the custom loss function to force the GRU to respect extreme historical crashes and rallies.

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.stats import genpareto
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

## 1. Historical Baseline & Dual GPD Fitting
We begin by downloading historical S&P 500 data to use as our baseline. We slice this data into 4-day rolling windows to train the GRU's short-term memory. 

Crucially, we use Extreme Value Theory (EVT) to isolate the worst 5% of historical crashes and the best 5% of historical rallies. We fit a Generalized Pareto Distribution (GPD) to both of these tails. These mathematical parameters will be used in our custom loss function later to teach the model about extreme tail-risk.

In [ ]:
print("Downloading baseline data...")
df = yf.download("^GSPC", start="2007-01-01", end="2021-12-31")['Close']
historical_returns = df.pct_change().dropna().to_numpy().flatten()

print("Preparing Training Data...")
X = [historical_returns[i: i + 4] for i in range(len(historical_returns) - 5)]
y = [historical_returns[i + 4: i + 5] for i in range(len(historical_returns) - 5)]

X_tensor = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)
y_tensor = torch.tensor(np.array(y), dtype=torch.float32)

split = int(len(X_tensor) * 0.9)
train_loader = DataLoader(
    TensorDataset(X_tensor[:split], y_tensor[:split]),
    batch_size=512,
    shuffle=True, # Critical: Prevents catastrophic forgetting of historical regimes
    drop_last=True
)
test_vector_set = X_tensor[split:]

print("Fitting Dual Tail Distributions (GPD)...")
tau = 0.05

# --- LEFT TAIL (Crashes) ---
threshold_left = np.quantile(historical_returns, tau)
extreme_left_tail = historical_returns[historical_returns <= threshold_left]
exceedances_left = -(extreme_left_tail - threshold_left)
shape_l, loc_l, scale_l = genpareto.fit(exceedances_left)
print(f"Left GPD (Crashes) - Shape: {shape_l:.4f}, Loc: {loc_l:.4f}, Scale: {scale_l:.4f}")

# --- RIGHT TAIL (Rallies) ---
threshold_right = np.quantile(historical_returns, 1 - tau)
extreme_right_tail = historical_returns[historical_returns >= threshold_right]
exceedances_right = extreme_right_tail - threshold_right
shape_r, loc_r, scale_r = genpareto.fit(exceedances_right)
print(f"Right GPD (Rallies) - Shape: {shape_r:.4f}, Loc: {loc_r:.4f}, Scale: {scale_r:.4f}")

# Convert parameters to tensors for PyTorch loss function
t_shape_l = torch.tensor(shape_l, dtype=torch.float32)
t_loc_l = torch.tensor(loc_l, dtype=torch.float32)
t_scale_l = torch.tensor(scale_l, dtype=torch.float32)
t_threshold_left = torch.tensor(threshold_left, dtype=torch.float32)

t_shape_r = torch.tensor(shape_r, dtype=torch.float32)
t_loc_r = torch.tensor(loc_r, dtype=torch.float32)
t_scale_r = torch.tensor(scale_r, dtype=torch.float32)
t_threshold_right = torch.tensor(threshold_right, dtype=torch.float32)

## 2. The Simple Generative Map (GRU)
Here we define the neural network architecture. It is a standard 2-layer Gated Recurrent Unit (GRU). It takes a 4-day sequence of returns (1 feature per day) and maps it to a single day's prediction. 

In [ ]:
class EtaMapGenerator(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, output_size=1, num_layers=2, dropout=0.2):
        super(EtaMapGenerator, self).__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.fc(out[:, -1, :])
        return out

model = EtaMapGenerator()

## 3. The Dual-Generative Loss Function
Standard Mean Squared Error (MSE) forces models to predict the average (leading to flat lines). To fix this, we combine MSE with a Wasserstein (W1) penalty. 

For every batch, we isolate the model's most extreme predictions. We map these predictions against dynamic empirical quantiles of our fitted Generalized Pareto Distributions. If the model fails to predict realistically severe crashes and rallies in a given batch, the W1 penalty mathematically forces the weights to adjust.

In [ ]:
def gpd_icdf(q, loc, scale, shape):
    """Analytical Inverse CDF for the Generalized Pareto Distribution."""
    if shape != 0:
        return loc + (scale / shape) * ((1 - q) ** (-shape) - 1)
    else:
        return loc - scale * torch.log(1 - q)

def generative_eta_loss(synthetic_returns, naive_inputs, lambda_param):
    mse = torch.nn.functional.mse_loss(synthetic_returns, naive_inputs)

    # Isolate Tails
    tail_threshold_left_val = torch.quantile(synthetic_returns, tau)
    tail_preds_left = synthetic_returns[synthetic_returns <= tail_threshold_left_val]

    tail_threshold_right_val = torch.quantile(synthetic_returns, 1 - tau)
    tail_preds_right = synthetic_returns[synthetic_returns >= tail_threshold_right_val]

    W1_left = torch.tensor(0.0, device=synthetic_returns.device)
    W1_right = torch.tensor(0.0, device=synthetic_returns.device)

    # Process Left Tail (Crashes)
    if len(tail_preds_left) >= 2:
        tail_preds_l_sorted, _ = torch.sort(tail_preds_left.squeeze())
        K_l = len(tail_preds_l_sorted)
        q_grid_l = torch.linspace(K_l / (K_l + 1), 1 / (K_l + 1), K_l, device=synthetic_returns.device)
        theoretical_exceedances_l = gpd_icdf(q_grid_l, t_loc_l, t_scale_l, t_shape_l)
        tail_true_theoretical_l = t_threshold_left - theoretical_exceedances_l
        W1_left = torch.mean(torch.abs(tail_preds_l_sorted - tail_true_theoretical_l))

    # Process Right Tail (Rallies)
    if len(tail_preds_right) >= 2:
        tail_preds_r_sorted, _ = torch.sort(tail_preds_right.squeeze())
        K_r = len(tail_preds_r_sorted)
        q_grid_r = torch.linspace(1 / (K_r + 1), K_r / (K_r + 1), K_r, device=synthetic_returns.device)
        theoretical_exceedances_r = gpd_icdf(q_grid_r, t_loc_r, t_scale_r, t_shape_r)
        tail_true_theoretical_r = t_threshold_right + theoretical_exceedances_r
        W1_right = torch.mean(torch.abs(tail_preds_r_sorted - tail_true_theoretical_r))

    # Unified Lambda Weighting
    W1_total = W1_left + W1_right
    total_loss = mse + (lambda_param * W1_total)

    return total_loss, mse, W1_total

## 4. Training (Generator Calibration)
We train the model for 1200 epochs. To ensure stability, we train purely on MSE for the first 300 epochs. Once the baseline weights settle, we activate `lambda_param`, turning on the GPD tail-risk penalties to teach the model extreme volatility dynamics.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
epochs = 1200

for epoch in range(epochs):
    for bx, by in train_loader:
        optimizer.zero_grad()
        pred = model(bx)
        
        # Delayed activation of the W1 penalty
        current_lambda = 0.0 if epoch < 300 else 0.5
        
        loss, mse, W1 = generative_eta_loss(pred, by, current_lambda)
        loss.backward()
        optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:04d} | Total Loss: {loss.item():.6f} | MSE: {mse.item():.6f} | W1_Total: {W1.item():.6f} | Lambda: {current_lambda}")

## 5. Generate Synthetic Stress-Test Paths (Stochastic Mixture)
To generate realistic future paths, we step the model forward one day at a time, feeding its own outputs back into its input sequence. 

To prevent the GRU from autoregressing into a flatline, we inject two types of mathematical noise at every time step:
1. **Daily Friction (Normal):** A tiny baseline vibration added to standard days.
2. **News Shocks (Lognormal):** A multiplier applied to random days to simulate sudden repricing events, forcing the GRU to react to sudden trauma. 

We apply a hard clip (`torch.clamp`) as a safety net to prevent any mathematical hallucinations from compounding to infinity.

In [ ]:
model.eval()

# Set duration of test
simulation_days = 252
num_paths = 10
starting_price = 2500.0

with torch.no_grad():
    random_indices = torch.randint(low=0, high=len(test_vector_set), size=(num_paths,))
    input_seq = test_vector_set[random_indices]
    synthetic_return_set = []

    testing_day = 0
    while testing_day < simulation_days:
        testing_day += 1

        # 1. The Raw Prediction
        pred_next_day = model(input_seq)

        # 2. Hard Clamp to prevent Autoregressive Explosion
        pred_next_day = torch.clamp(pred_next_day, min=-0.15, max=0.15)

        # 3. Probability Matrix (Dictates which paths get standard noise vs news shocks today)
        p_matrix = torch.rand(num_paths, 1)
        normal_mask = p_matrix > 0.3
        lognormal_mask = p_matrix <= 0.3

        # 4. RNG 1: Daily Friction (Normal Dist)
        if normal_mask.any():
            normal_noise = torch.normal(mean=0.0, std=0.0001, size=(num_paths, 1))
            pred_next_day[normal_mask] = pred_next_day[normal_mask] + normal_noise[normal_mask]

        # 5. RNG 2: News Shocks (Lognormal Dist)
        if lognormal_mask.any():
            lognormal_noise = torch.zeros(num_paths, 1).log_normal_(mean=0.0, std=0.03)
            gross_returns = 1.0 + pred_next_day[lognormal_mask]
            shocked_returns = (gross_returns * lognormal_noise[lognormal_mask]) - 1.0
            pred_next_day[lognormal_mask] = shocked_returns

        synthetic_return_set.append(pred_next_day)

        # 6. Roll Forward Memory (Shift out the oldest day, append the newest)
        input_seq = torch.roll(input_seq, shifts=-1, dims=1)
        input_seq[:, -1, :] = pred_next_day

    synthetic_return_set = torch.stack(synthetic_return_set).squeeze(-1).numpy()

price_paths = starting_price * np.cumprod(1 + synthetic_return_set, axis=0)

# Plotting the 1-Year Stress Tests
plt.figure(figsize=(12, 6))
plt.plot(price_paths, alpha=0.3, linewidth=1)
plt.title(f"{num_paths} Synthetic {simulation_days // 252}-Year Stock Paths (Dual-Tail + Stochastic Mixture)")
plt.xlabel("Trading Days")
plt.ylabel("Simulated Price ($)")
plt.grid(alpha=0.3)
plt.show()